## 18. 结构化输出：让 agent 返回验证过的 JSON

> 来源：[Get structured output from agents](https://code.claude.com/docs/en/agent-sdk/structured-outputs)

自由文本适合 chat，不适合程序消费。`output_format` 让 agent 随便用工具干活，**最终必须返回符合你 JSON Schema 的数据**——SDK 负责校验，不匹配就自动 re-prompt 重试；重试耗尽则以 `error_max_structured_output_retries` 收场而不是给你脏数据。

```python
options = ClaudeAgentOptions(
    output_format={"type": "json_schema", "schema": <JSON Schema dict>}
)
# 结果在 ResultMessage.structured_output（dict）
```

支持的 Schema 特性：全部基本类型、`enum`、`const`、`required`、嵌套 object、`$ref`。不是全部 JSON Schema 特性都支持——完整清单见官方 platform 文档 "JSON Schema limitations"。

**Pydantic 是 Python 侧的最佳拍档**：`Model.model_json_schema()` 生成 schema，`Model.model_validate(msg.structured_output)` 把结果还原成带类型的对象——定义、校验、消费三步全类型安全。

错误处理按 subtype 分流：

```python
if isinstance(message, ResultMessage):
    if message.subtype == "success" and message.structured_output:
        ...  # 用验证过的数据
    elif message.subtype == "error_max_structured_output_retries":
        ...  # 检查 message.errors 区分失败原因：schema 验证失败，还是 model fallback 撤回输出
```

避错三条官方建议：schema 保持聚焦（深嵌套 + 大量 required 更难满足）；任务可能拿不到的信息设为 optional 字段；prompt 写清楚要输出什么。

下方两个 cell：第一个是 Pydantic 全类型安全版；第二个是官方的 TODO 采集 agent——手写 schema、optional 字段与多步工具调用的配合（agent 先 Grep 再 git blame，最后把结果整理成符合 schema 的 JSON 返回；blame 拿不到的 author/date 没进 `required`，缺了也不会验证失败）。

In [ ]:
from pydantic import BaseModel
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


class Step(BaseModel):
    step_number: int
    description: str
    estimated_complexity: str  # 'low' / 'medium' / 'high'


class FeaturePlan(BaseModel):
    feature_name: str
    summary: str
    steps: list[Step]
    risks: list[str]


async def demo_structured_output():
    async for message in query(
        prompt="Plan how to add dark mode support to a React app. Break it into implementation steps.",
        options=ClaudeAgentOptions(
            output_format={
                "type": "json_schema",
                "schema": FeaturePlan.model_json_schema(),  # Pydantic 生成 schema
            }
        ),
    ):
        if isinstance(message, ResultMessage):
            if message.subtype == "success" and message.structured_output:
                plan = FeaturePlan.model_validate(
                    message.structured_output
                )  # 还原为类型对象
                print(f"Feature: {plan.feature_name}")
                for step in plan.steps:
                    print(
                        f"{step.step_number}. [{step.estimated_complexity}] {step.description}"
                    )
            elif message.subtype == "error_max_structured_output_retries":
                print("Could not produce valid output:", message.errors)


await demo_structured_output()

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

# 手写 schema：author/date 故意不进 required——git blame 拿不到也不会验证失败
TODO_SCHEMA = {
    "type": "object",
    "properties": {
        "todos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "file": {"type": "string"},
                    "line": {"type": "integer"},
                    "text": {"type": "string"},
                    "author": {"type": "string"},
                    "date": {"type": "string"},
                },
                "required": ["file", "line", "text"],
            },
        },
        "total_count": {"type": "integer"},
    },
    "required": ["todos", "total_count"],
}


async def demo_todo_agent():
    # agent 自主跑多步工具（Grep 找 TODO、Bash 跑 git blame），最后按 schema 整理成 JSON 返回
    async for message in query(
        prompt="Find all TODO comments in this codebase. Use git blame to find "
               "who added each one and when.",
        options=ClaudeAgentOptions(
            allowed_tools=["Grep", "Bash", "Read"],
            output_format={"type": "json_schema", "schema": TODO_SCHEMA},
        ),
    ):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            data = message.structured_output or {}
            for todo in data.get("todos", []):
                who = todo.get("author", "?")   # optional 字段用 .get 兜底
                print(f"{todo['file']}:{todo['line']}  {todo['text']}  (by {who})")
            print("total:", data.get("total_count"))


await demo_todo_agent()